<a href="https://colab.research.google.com/github/rudra629/ml-internship-flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

My lane is Refresh / Content Opportunity Scoring. As an ML task, this is framed as Binary Classification. We are categorizing web pages into two distinct buckets: those that urgently need a human content review (1) and those that do not (0), based on their current search performance signals.

The target label is needs_refresh. Because we don't have historical data on which pages actually got rewritten in the past, we define a proxy target: a page is flagged as needing a refresh (1) if it has above-median impressions (high visibility) but below-median Click-Through Rate (CTR). This proxies the concept of "wasted visibility."

The primary success metric is Precision. Since the action involves deploying human effort (a writer or SEO spending 20-30 minutes auditing a page), false positives (telling them to review a perfectly fine page) waste expensive human time. We want to be highly confident when the model flags a page for review.

In [1]:
import duckdb
import pandas as pd
import getpass


hf_token = getpass.getpass("Enter your Hugging Face read token: ")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")


REL = 'hf://datasets/FlyRank/internship-warehouse'
query = f"""
SELECT
    content_hash_id as unit_of_analysis,
    SUM(gsc_clicks) as total_clicks,
    SUM(gsc_impressions) as total_impressions,
    AVG(gsc_avg_position) as avg_position,
    (SUM(gsc_clicks)*1.0 / NULLIF(SUM(gsc_impressions), 0)) as ctr
FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
GROUP BY content_hash_id
HAVING total_impressions > 1000
LIMIT 5
"""
df = con.sql(query).df().dropna()


ctr_threshold = df['ctr'].median()
df['needs_refresh_target'] = ((df['total_impressions'] > df['total_impressions'].median()) &
                              (df['ctr'] < ctr_threshold)).astype(int)

df.head()

Enter your Hugging Face read token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,unit_of_analysis,total_clicks,total_impressions,avg_position,ctr,needs_refresh_target
0,content_d5479697a0828b33,31.0,6115.0,33.292638,0.005070,0
1,content_d7daac9d28863fda,65.0,12182.0,14.182897,0.005336,0
2,content_b313be479d6d3707,130.0,16870.0,12.537416,0.007706,0
3,content_5175438fecb054a4,147.0,21688.0,12.718705,0.006778,0
4,content_3a204f16288e29ed,8.0,2535.0,20.989052,0.003156,0


A fixed rule (like "review anything past position 10") is too rigid and ignores context. CTR decay varies non-linearly depending on the position and the specific SERP (Search Engine Results Page). A machine learning model—like a Decision Tree—can learn the complex, multi-variable relationships between average position, impression volume, and expected CTR, isolating genuine underperformance better than an arbitrary, hardcoded threshold.

[x] Named the ML task type, target/proxy, and success metric

[x] Showed the unit of analysis as a real dataframe

[x] Explained why this is an ML/analysis problem and not just a fixed rule

[x] Tied the output to a real content action